In [ ]:
import pandas as pd
from pathlib import Path

# Figure out project root (works whether you run from / or /notebooks)
cwd = Path.cwd()
BASE_DIR = cwd.parent if cwd.name.lower() == "notebooks" else cwd

raw_path = BASE_DIR / "data" / "KorCCVi_v2.csv"
out_path = BASE_DIR / "data" / "transcripts.csv"

print("Project root:", BASE_DIR)
print("Reading:", raw_path)

df = pd.read_csv(raw_path)
print("Original columns:", df.columns.tolist())
print(df.head())

# ---- 1. Normalise column names ----
df = df.rename(columns={
    "Transcript": "transcript",
    "transcript": "transcript",
    "Label": "label",
    "label": "label",
    "ID": "id",
    "Id": "id"
})

if "id" not in df.columns:
    df["id"] = range(1, len(df) + 1)

# Drop rows with missing text or label
df = df.dropna(subset=["transcript", "label"]).copy()

# ---- 2. Map labels 0/1 -> 'safe' / 'vishing' ----
label_map = {1: "vishing", 0: "safe", "1": "vishing", "0": "safe"}
df["label_str"] = df["label"].map(label_map)

unknown = df["label_str"].isna().sum()
if unknown > 0:
    print(f"Warning: {unknown} rows had unknown labels and were dropped.")
    df = df.dropna(subset=["label_str"])

# ---- 3. Synthetic filename ----
df["filename"] = df["id"].apply(lambda x: f"korccvi_{int(x)}.wav")

final_df = df[["filename", "label_str", "transcript"]].rename(
    columns={"label_str": "label"}
)

print("\nLabel distribution:")
print(final_df["label"].value_counts())

print("\nSample rows:")
print(final_df.head())

# Optional: shuffle rows
final_df = final_df.sample(frac=1.0, random_state=42).reset_index(drop=True)

# ---- 4. Save to transcripts.csv that your training code uses ----
out_path.parent.mkdir(exist_ok=True, parents=True)
final_df.to_csv(out_path, index=False, encoding="utf-8-sig")

print(f"\nSaved merged transcripts to: {out_path}")
print("Total samples:", len(final_df))
